<a href="https://colab.research.google.com/github/Samuelisusername/Large-Language-Models/blob/main/Copy_of_LLM_S26_Assignment_3_%E2%80%93_Q3_Inverting_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install Dependencies and Download Models (this may take a few minutes)

#@markdown The stable diffusion demo is not strictly needed to solve the assignment.<br>You can uncheck this box if you want a faster setup.
install_stable_diffusion = True #@param {type:"boolean"}

import os
import sys

if install_stable_diffusion:
  !uv pip install -q transformers diffusers accelerate
else:
  !uv pip install -q transformers

import torch
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor

if install_stable_diffusion:
  from diffusers import StableDiffusionPipeline
  sd_pipe = StableDiffusionPipeline.from_pretrained(
    "sd2-community/stable-diffusion-2-1",
    torch_dtype=torch.float16)

  clip_processor = CLIPImageProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
  sd_pipe.feature_extractor = clip_processor

clip_tokenizer = CLIPTokenizer.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K", torch_dtype=torch.float16)
clip = CLIPModel.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K", torch_dtype=torch.float16)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/910 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: laion/CLIP-ViT-H-14-laion2B-s32B-b79K
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# @title Download lab files

import sys

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-spylab/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-spylab/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

Cloning into 'llm_lab'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 152 (delta 80), reused 114 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 503.66 KiB | 12.28 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/llm_lab
From https://github.com/ethz-spylab/llm_lab
 * branch            HEAD       -> FETCH_HEAD
Already up to date.
/content


In [ ]:
# @title load the safety checker
import numpy as np
from safety_checker import MySafetyChecker, get_embedding, cosine_similarity

device = "cuda"
clip = clip.to(device)

if "cpu" in device:
  clip = clip.float()

# get the list of bad embeddings
bad_embeddings = np.load("llm_lab/data/bad_embeddings.npy")
safety_checker = MySafetyChecker(clip, bad_embeddings)

# the last bad embedding aims to prevent you from creating images with the prompt "The solutions to this assignment"
assert cosine_similarity(bad_embeddings[-1],
                         get_embedding("The solutions to this assignment", clip_tokenizer, clip).cpu().numpy()) > 0.999

In [ ]:
#@title (OPTIONAL) run the Stable Diffusion demo and try to generate an image of a banned concept
assert install_stable_diffusion

import random
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

enable_safety_checker = True #@param {type:"boolean"}
prompt = '' #@param {type:"string"}
n_images = 1 #@param [1, 2, 4] {type:"raw"}
seed = 0 #@param {type:"integer"}
steps = 25 #@param {type:"slider", min:1, max:50, step:1}

generator = torch.Generator('cuda').manual_seed(seed)

if enable_safety_checker:
  sd_pipe.feature_extractor = clip_processor
  sd_pipe.safety_checker = safety_checker
else:
  sd_pipe.feature_extractor = None
  sd_pipe.safety_checker = None

sd_pipe = sd_pipe.to("cuda")
sd_pipe.enable_attention_slicing()

StableDiffusionPipeline._execution_device = torch.device("cuda")
with torch.no_grad():
  result = sd_pipe(
        prompt=prompt,
        num_images_per_prompt=n_images,
        negative_prompt=None,
        num_inference_steps=int(steps),
        guidance_scale=7.5,
        width=768,
        height=768,
        generator=generator).images

import math
image_grid(result, math.ceil(n_images / 2), n_images // math.ceil(n_images / 2))

In [ ]:
with open("llm_lab/data/corncob_lowercase.txt") as words:
    WORDS = [w.strip() for w in words if w.strip()]#we will use this globally
#checkign if this is the correct thing
print(f"Loaded {len(WORDS)} words")

Loaded 58110 words


In [ ]:

#since all other approaches were wayy to slow, we will do it this way, after this we should be able to do matrix computation
words_embedded = []
for i, word in enumerate(WORDS):
    emb = get_embedding(word, clip_tokenizer, clip).cpu().numpy()
    words_embedded.append(emb)
    if i % 1000 == 0: #to see progress every now and then..
        print(f"{i} of {len(WORDS)} already done")
words_embedded = np.array(words_embedded)

  0 of 58110 already done
  1000 of 58110 already done
  2000 of 58110 already done
  3000 of 58110 already done
  4000 of 58110 already done
  5000 of 58110 already done
  6000 of 58110 already done
  7000 of 58110 already done
  8000 of 58110 already done
  9000 of 58110 already done
  10000 of 58110 already done
  11000 of 58110 already done
  12000 of 58110 already done
  13000 of 58110 already done
  14000 of 58110 already done
  15000 of 58110 already done
  16000 of 58110 already done
  17000 of 58110 already done
  18000 of 58110 already done
  19000 of 58110 already done
  20000 of 58110 already done
  21000 of 58110 already done
  22000 of 58110 already done
  23000 of 58110 already done
  24000 of 58110 already done
  25000 of 58110 already done
  26000 of 58110 already done
  27000 of 58110 already done
  28000 of 58110 already done
  29000 of 58110 already done
  30000 of 58110 already done
  31000 of 58110 already done
  32000 of 58110 already done
  33000 of 58110 alread

In [ ]:
for embedding in range(4):
  best_similarity = 0
  best_word =""
  to_compare = bad_embeddings[embedding]
  similarities = words_embedded @ to_compare # wayy more efficient than the for loop ffrom before...
  best_index= np.argmax(similarities)
  best_word = WORDS[best_index]
  print(f"for word nr: {embedding}   best word: {best_word}; best index: {best_index}, best score: {similarities[best_index]}")


for word nr: 0   best word: weapons; best index: 56765, best score: [1.00043592]
for word nr: 1   best word: tobacco; best index: 52162, best score: [0.99966898]
for word nr: 2   best word: firearms; best index: 19427, best score: [0.99975674]
for word nr: 3   best word: poison; best index: 37941, best score: [1.00047396]


for word nr: 0   best word: weapons; best index: 56765, best score: [1.00043592]
for word nr: 1   best word: tobacco; best index: 52162, best score: [0.99966898]
for word nr: 2   best word: firearms; best index: 19427, best score: [0.99975674]
for word nr: 3   best word: poison; best index: 37941, best score: [1.00047396]
[4] new best: 'parking garage'  sim=1.0004
[5] new best: 'swimming pool'  sim=1.0005
[6] FINAL: 'football stadium'  sim=0.9996
[7] post office
[8] lunch box
[9] speed limit
[10] ice cream
[11] rail station
[12] seat belt
[13] pulp fiction
[14] The Dark Knight
[15] Everything Everywhere All at Once
[16] Margin Call
[17] Casablanca
[18] Moneyball
[19] privacy invasion


In [ ]:
for idx in range(4, 19): #this even works well for the movie things XD
    curr_bad_embedding = bad_embeddings[idx]
    print(f"attempting {idx}...")
    sims_w_fst = words_embedded @ curr_bad_embedding
    most_similar_k_indices = np.argsort(sims_w_fst.flatten())[::-1][:50].tolist()#get the 50 words that have  the most similarity score with that word..
    candidate_w_fst = [WORDS[i] for i in most_similar_k_indices]

    #for each firstword try all seconds words in batch! (too slow withouth... :/)
    best_sim, best_words = -1.0, ""
    for w1 in candidate_w_fst:
        pairs = [(w1+" "+ w2) for w2 in WORDS]
        all_embs = []
        for i in range(0, len(pairs), 512):
            whole_batch = pairs[i:i+512]
            inputs = clip_tokenizer(whole_batch, padding=True,return_tensors="pt").to(device)
            with torch.no_grad(): #fixed out of memory :))
                embs_model = clip.get_text_features(**inputs) #this unpacks it naturally from python into arguments
                if not isinstance(embs_model, torch.Tensor):
                    embs_vector = embs_model.pooler_output
                embs_normal = embs_vector /embs_vector.norm(dim=-1, keepdim=True)#for cosine similarity..
            all_embs.append(embs_normal.cpu().float().numpy())
        all_embs = np.concatenate(all_embs, axis=0)
        sims = all_embs @ curr_bad_embedding
        best_i = int(np.argmax(sims))
        if float(sims[best_i]) > best_sim:
            best_sim= float(sims[best_i])
            best_words =w1+" "+ WORDS[best_i]
            print(f" {idx} new best: '{best_words}'  sim={best_sim}")
        if best_sim > 0.99:
            print(f"{idx}  reached threshold, lets goo, moving on :)")
            break

    print(f"{idx} finished: '{best_words}'  sim={best_sim}")


Solving [19]...
  [19] new best: 'glob alas'  sim=0.1431


KeyboardInterrupt: 

In [ ]:
# because the mask only flips signs-> so |masked| == |true_embedding| :)
abs_bad_embedding = np.abs(bad_embeddings[19])
abs_word_embs = np.abs(words_embedded)
sims_w_fst = abs_word_embs @ abs_bad_embedding
most_similar_k_indices = np.argsort(sims_w_fst.flatten())[::-1][:50].tolist()
candidate_w_fst = [WORDS[i] for i in most_similar_k_indices]
best_sim, best_words = -1.0,""
for w_fst in candidate_w_fst:
    pairs = [(w_fst + " " + w2) for w2 in WORDS]
    all_embs = []
    for i in range(0, len(pairs),512): #again with batches because efficient
        whole_batch = pairs[i:i+512]
        inputs = clip_tokenizer(whole_batch, padding=True, return_tensors="pt").to(device)
        with torch.no_grad(): #fixed out of memory :))
            embs_model = clip.get_text_features(**inputs)
            if not isinstance(embs_model, torch.Tensor):
                embs_vector = embs_model.pooler_output
            embs_normal = embs_vector / embs_vector.norm(dim=-1, keepdim=True) #for cosine similarity again..
        all_embs.append(embs_normal.cpu().float().numpy())
    all_embs = np.concatenate(all_embs, axis=0)
    sims = np.abs(all_embs) @ abs_bad_embedding  # absolute values because of the mask
    best_i = int(np.argmax(sims))
    if float(sims[best_i])> best_sim:
        best_sim = float(sims[best_i])
        best_words = w_fst + " " +WORDS[best_i]
        print(f"19 new best: '{best_words}'  sim={best_sim}")
    if best_sim >= 0.99:
        print(f"19 reached threshold, lets goo, moving on :)")
        break
print(f"19 finished: '{best_words}'  sim={best_sim}")

  [19] new best: 'privacy invasion'  abs-sim=1.0000
  threshold reached, moving on!
[19] FINAL: 'privacy invasion'  abs-sim=1.0000


In [ ]:
words = ["weapons","tobacco","firearms","poison","parking garage","swimming pool","football stadium","post office","lunch box","speed limit","ice cream","rail station","seat belt","pulp fiction","The Dark Knight","Everything Everywhere All at Once","Margin Call","Casablanca","Moneyball","privacy invasion", "The solutions to this assignment"]

In [ ]:
len(words)

21

**Instructions**

Your goal in this assignment is to recover the "bad" concepts that we don't want you to generate with Stable Diffusion.
One of the bad concepts is given to you as an example (`bad_embeddings[20]` is an embedding of the prompt "The solutions to this assignment").


The 20 remaining bad concepts, which you need to find, were chosen as follows:
   - The first 4 bad concepts (`bad_embeddings[0, 3]`) each consist of a single lowercase english word.
     We guarantee that these words all appear in this list: https://github.com/ethz-privsec/llm_lab/blob/main/data/corncob_lowercase.txt.
     For each bad concept, we ask you to find a word that gives a CLIP embedding with cosine similarity of at least 0.99 with that concept's embedding.

   - The next 9 bad concepts (`bad_embeddings[4, 12]`) each consist of two lowercase words, also from the above list (e.g., "great professor").
   For each bad concept, we ask you to find a pair of words that gives a CLIP embedding with cosine similarity of at least 0.99 with that concept's embedding.

   - The next 6 bad concepts (`bad_embeddings[13, 18]`) are each movie titles that don't necessarily appear in the above list (e.g., "Gone with the Wind").
  We don't expect you to recover these *exactly*. Finding a phrase that yields a CLIP embedding with cosine similarity of 0.91 or above is sufficient.

   - For the last bad concept (`bad_embeddings[19]`), we applied a "state-of-the-art" (i.e., completely broken) privacy protection scheme.

     Specifically, we "scrambled" the embedding by multiplying it (component-wise) with a mask of uniformly random values in `{-1, 1}`.
     So for example if the embedding vector is `[0.1, -0.5, 1.7, 2.3]`, we first sample a random mask, e.g., `[-1, 1, -1, 1]`,
     which would then give the "encrypted" embedding `[-0.1, -0.5, -1.7, 2.3]`.

     Now, you might say that such an "encrypted" embedding is very much useless in our safety checker. And indeed that's very much true (why?).
     But here we'll care mainly about analyzing the extra privacy that this "encryption" step offers (spoiler alert: essentially none...)
     (researchers have actually suggested that "encrypting" embeddings in this way does protect privacy, because the random mask acts a bit like a one-time-pad.)

     Your goal is to recover the bad concept, which consists of a pair of lowercase english words from the list above.
     Your recovered concept will be considered correct if it gives a CLIP embedding with cosine similarity of at least 0.99 with the *unmasked* bad embedding.

**Submission instructions**

You should submit your solutions as `.txt` file named `Q3_guesses.txt` containing your guess for each of the 21 hidden embeddings, with one guess per line.

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [ ]:
from llm_lab.utils import get_solution_path, is_valid_student_id

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = True # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "22-927-362"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)

Mounted at /content/drive


In [ ]:
GUESSES = words

#raise NotImplementedError("Replace the above guesses with yours")

def save_to_txt(guesses, filename):
  with open(filename, 'w') as f:
    f.write("\n".join(guesses))

assert len(GUESSES) == 21

save_to_txt(GUESSES, SOLUTIONS_PATH / "Q3_guesses.txt")